# MobileNetV3 Transfer Learning — Flower Classification (PyTorch)

**Model:** MobileNetV3-Small / MobileNetV3-Large (pretrained on ImageNet)  
**Dataset:** Oxford Flowers 102 (102 flower species)  
**Focus:** Lightweight, mobile/edge-friendly classification

**MobileNetV3 Key Features:**
- Inverted residual blocks (MBConv) + Squeeze-and-Excitation (SE)
- h-swish activation (hardware-friendly)
- Neural Architecture Search (NAS) optimized
- Small variant (~2.5M params) / Large variant (~5.4M params)

**Architecture:**
```
Input (3 × 224 × 224)
    ↓
MobileNetV3 Backbone (pretrained ImageNet)
    ↓
AdaptiveAvgPool → Flatten
    ↓
FC(960/576 → 1280) → Hardswish → Dropout(0.2)   ← original classifier
    ↓  (replaced with ↓)
FC(1280 → 256) → ReLU → Dropout(0.3) → FC(256 → 102)
```

> GPU accelerator ဖွင့်ပြီး run ပါ: `Settings → Accelerator → GPU`

In [ ]:
# --- 1. Install Dependencies ---
# !pip install -q torch torchvision datasets scikit-learn seaborn tqdm

In [ ]:
# --- 2. Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import time
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("✅ All imports successful.")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- 3. Device & Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hyperparameters
IMG_SIZE = 224          # MobileNetV3 standard input
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-5
NUM_WORKERS = 2
FREEZE_EPOCHS = 5
NUM_CLASSES = 102       # Oxford Flowers 102

# MobileNetV3 variant — 'small' or 'large'
VARIANT = 'large'  # Change to 'small' for smaller model

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"MobileNetV3 variant: {VARIANT}")
print(f"Num classes: {NUM_CLASSES}")

## 📦 Dataset — Oxford Flowers 102

Oxford Flowers 102 dataset ကို HuggingFace `datasets` library နဲ့ download လုပ်ပါမယ်။  
- 102 flower species, ~8,000 images total  
- Fine-grained classification task (MobileNetV3 ရဲ့ lightweight capability test)

In [ ]:
# --- 4. Download Oxford Flowers 102 ---
from datasets import load_dataset

print("⏳ Downloading Oxford Flowers 102...")
hf_dataset = load_dataset("nelorth/oxford-flowers")
print(f"✅ Download complete!")
print(f"Train: {len(hf_dataset['train'])} | Test: {len(hf_dataset['test'])}")

# Get class names (flower species)
FLOWER_NAMES = hf_dataset['train'].features['label'].names
print(f"Classes: {NUM_CLASSES}")
print(f"Sample classes: {FLOWER_NAMES[:10]}...")

In [ ]:
# --- 5. Custom Dataset Class ---
class FlowersDataset(Dataset):
    """Wraps HuggingFace dataset for PyTorch DataLoader."""
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

# --- 6. Data Transforms ---
# ImageNet normalization (MobileNetV3 pretrained stats)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),       # 256
    transforms.CenterCrop(IMG_SIZE),         # 224
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# --- 7. Create Datasets & DataLoaders ---
# Split train into train + val (85/15)
full_train = hf_dataset['train']
total = len(full_train)
val_size = int(0.15 * total)
train_size = total - val_size

# Use random indices for split
indices = torch.randperm(total).tolist()
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_subset = full_train.select(train_indices)
val_subset   = full_train.select(val_indices)
test_data    = hf_dataset['test']

train_dataset = FlowersDataset(train_subset, transform=train_transform)
val_dataset   = FlowersDataset(val_subset,   transform=val_transform)
test_dataset  = FlowersDataset(test_data,    transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"📂 Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"📦 Batches → Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# --- 8. Visualize Sample Images ---
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for display."""
    t = tensor.clone()
    for ch, m, s in zip(t, mean, std):
        ch.mul_(s).add_(m)
    return torch.clamp(t, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(FLOWER_NAMES[labels[i]], fontsize=9, fontweight='bold')
    ax.axis('off')
fig.suptitle("🌸 Sample Training Images — Oxford Flowers 102", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Class distribution
print("\n📊 Class Distribution:")
train_labels = [train_subset[i]['label'] for i in range(len(train_subset))]
label_counts = Counter(train_labels)

fig, ax = plt.subplots(figsize=(20, 4))
sorted_counts = sorted(label_counts.items())
ax.bar([x[0] for x in sorted_counts], [x[1] for x in sorted_counts], color='coral', alpha=0.8)
ax.set_xlabel("Class Index")
ax.set_ylabel("Count")
ax.set_title("Training Set — Class Distribution")
plt.tight_layout()
plt.show()
print(f"Min samples/class: {min(label_counts.values())} | Max: {max(label_counts.values())}")

## 🏗️ Model — MobileNetV3 Transfer Learning

### Architecture Modifications:
```
MobileNetV3-Large (Pretrained ImageNet)
├── features (Inverted Residual Blocks + SE modules) → FROZEN in Phase 1
└── classifier (REPLACED)
    ├── Linear(960 → 512)
    ├── Hardswish + BatchNorm + Dropout(0.3)
    ├── Linear(512 → 256)
    ├── Hardswish + BatchNorm + Dropout(0.2)
    └── Linear(256 → 102)
```

### MobileNetV3 Key Innovations:
| Feature | Description |
|---------|-------------|
| **h-swish** | Hardware-friendly Swish: `x · ReLU6(x+3)/6` |
| **SE Block** | Squeeze-and-Excitation for channel attention |
| **NAS + NetAdapt** | Architecture search + layer-wise tuning |
| **Inverted Residuals** | Expand → Depthwise Conv → Squeeze (from MobileNetV2) |
| **Last Stage Redesign** | Moved expensive layers after pooling |

In [ ]:
# --- 9. Build MobileNetV3 Model ---
def build_mobilenetv3(variant='large', num_classes=102, dropout=0.3):
    """
    Load pretrained MobileNetV3 and replace classifier head.

    Args:
        variant: 'large' or 'small'
        num_classes: output classes
        dropout: dropout rate for classifier
    """
    if variant == 'large':
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        model = models.mobilenet_v3_large(weights=weights)
        in_features = 960  # Large variant last channel
    else:
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=weights)
        in_features = 576  # Small variant last channel

    # Replace classifier head
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(inplace=True),
        nn.BatchNorm1d(512),
        nn.Dropout(p=dropout),
        nn.Linear(512, 256),
        nn.Hardswish(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=dropout * 0.7),
        nn.Linear(256, num_classes),
    )

    # Initialize new classifier weights
    for m in model.classifier.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    return model

model = build_mobilenetv3(variant=VARIANT, num_classes=NUM_CLASSES)
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📐 MobileNetV3-{VARIANT.capitalize()}")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Model size:       ~{total_params * 4 / 1024**2:.1f} MB (FP32)")
print(f"\n🔧 Classifier Head:")
print(model.classifier)

In [ ]:
# --- 10. Freeze Backbone for Phase 1 ---
def freeze_backbone(model):
    """Freeze all feature extraction layers."""
    for param in model.features.parameters():
        param.requires_grad = False
    print("🔒 Backbone FROZEN — training classifier head only")

def unfreeze_backbone(model):
    """Unfreeze all layers for fine-tuning."""
    for param in model.features.parameters():
        param.requires_grad = True
    print("🔓 Backbone UNFROZEN — fine-tuning all layers")

# Phase 1: Freeze backbone
freeze_backbone(model)

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable params (Phase 1): {trainable_now:,} / {total_params:,}")
print(f"   Frozen: {total_params - trainable_now:,} params")

## 🎯 Training — 2-Phase Strategy

### Phase 1: Feature Extraction (Backbone Frozen)
- Only classifier head trains → fast convergence, no catastrophic forgetting
- Higher learning rate (1e-3), `FREEZE_EPOCHS=5`

### Phase 2: Fine-Tuning (Full Model)
- Unfreeze backbone → differential learning rates
- Backbone: `1e-5` (preserve pretrained features)
- Classifier: `5× backbone LR` (continue adaptation)
- `ReduceLROnPlateau` + `CosineAnnealingWarmRestarts`

In [ ]:
# --- 11. Loss, Optimizer & Scheduler ---
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Phase 1 optimizer — only classifier params
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

print(f"⚙️  Loss:      CrossEntropyLoss (label_smoothing=0.1)")
print(f"⚙️  Optimizer: Adam (lr={LEARNING_RATE}, wd=1e-4)")
print(f"⚙️  Scheduler: ReduceLROnPlateau (patience=2, factor=0.5)")
print(f"⚙️  AMP:       {'Enabled ✅' if scaler else 'Disabled (CPU)'}")

In [ ]:
# --- 12. Training & Validation Functions ---
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with optional mixed precision."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate model on given loader."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

print("✅ Training & validation functions defined")

In [ ]:
# --- 13. Full Training Loop (2-Phase) ---
SAVE_PATH = "best_mobilenetv3_flowers102.pth"

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'lr': [],         'phase': [],
}

best_val_acc = 0.0
total_epochs = FREEZE_EPOCHS + NUM_EPOCHS

print("=" * 70)
print(f"🚀 PHASE 1: Feature Extraction (Epochs 1-{FREEZE_EPOCHS})")
print(f"   Backbone FROZEN | LR = {LEARNING_RATE}")
print("=" * 70)

for epoch in range(1, total_epochs + 1):
    # --- Phase Transition ---
    if epoch == FREEZE_EPOCHS + 1:
        print("\n" + "=" * 70)
        print(f"🔥 PHASE 2: Fine-Tuning (Epochs {FREEZE_EPOCHS+1}-{total_epochs})")
        print(f"   Backbone UNFROZEN | Differential LR")
        print("=" * 70)

        unfreeze_backbone(model)

        # New optimizer with differential learning rates
        optimizer = optim.Adam([
            {'params': model.features[:8].parameters(),  'lr': FINE_TUNE_LR * 0.1},   # Early layers
            {'params': model.features[8:].parameters(),   'lr': FINE_TUNE_LR},          # Late layers
            {'params': model.classifier.parameters(),     'lr': FINE_TUNE_LR * 5},      # Classifier
        ], weight_decay=1e-4)

        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=2, eta_min=1e-7
        )

    # --- Train & Validate ---
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    # Scheduler step
    current_lr = optimizer.param_groups[-1]['lr']
    if epoch <= FREEZE_EPOCHS:
        scheduler.step(val_loss)
    else:
        scheduler.step(epoch - FREEZE_EPOCHS)

    # Record history
    phase = 1 if epoch <= FREEZE_EPOCHS else 2
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    history['phase'].append(phase)

    # Save best model
    improved = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'variant': VARIANT,
            'num_classes': NUM_CLASSES,
            'flower_names': FLOWER_NAMES,
        }, SAVE_PATH)
        improved = " ⭐ BEST"

    print(f"  [{epoch:2d}/{total_epochs}] "
          f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.2e} | ⏱ {elapsed:.1f}s{improved}")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")
print(f"💾 Model saved to: {SAVE_PATH}")

In [ ]:
# --- 14. Training History Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

# Phase boundary
phase_boundary = FREEZE_EPOCHS + 0.5

# Loss curves
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
ax.plot(epochs_range, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('📉 Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy curves
ax = axes[1]
ax.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Train Acc')
ax.plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', markersize=3, label='Val Acc')
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('📈 Accuracy Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[2]
ax.plot(epochs_range, history['lr'], 'g-o', markersize=3)
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('📐 Learning Rate Schedule')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('MobileNetV3 Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 Evaluation — Test Set Performance

In [ ]:
# --- 15. Load Best Model & Test Evaluation ---
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (Val Acc: {checkpoint['val_acc']:.4f})")

# Full test evaluation
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
        else:
            outputs = model(images)

        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Correct: {(all_preds == all_labels).sum()} / {len(all_labels)}")

# Top-5 accuracy
top5_correct = 0
for i in range(len(all_labels)):
    top5 = np.argsort(all_probs[i])[-5:]
    if all_labels[i] in top5:
        top5_correct += 1
top5_acc = top5_correct / len(all_labels)
print(f"   Top-5 Accuracy: {top5_acc:.4f} ({top5_acc*100:.2f}%)")

In [ ]:
# --- 16. Classification Report ---
from sklearn.metrics import classification_report, confusion_matrix

# Full classification report (top 20 classes by support)
report = classification_report(all_labels, all_preds,
                               target_names=FLOWER_NAMES,
                               output_dict=True)
print("📋 Classification Report (Top 20 classes by F1-score):\n")

# Sort by F1 and show top 20
class_metrics = {k: v for k, v in report.items()
                 if k not in ['accuracy', 'macro avg', 'weighted avg']}
sorted_classes = sorted(class_metrics.items(), key=lambda x: x[1]['f1-score'], reverse=True)

print(f"{'Class':<30} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 75)
for name, metrics in sorted_classes[:20]:
    print(f"{name:<30} {metrics['precision']:>10.3f} {metrics['recall']:>10.3f} "
          f"{metrics['f1-score']:>10.3f} {int(metrics['support']):>10}")
print("-" * 75)
print(f"{'Macro Avg':<30} {report['macro avg']['precision']:>10.3f} "
      f"{report['macro avg']['recall']:>10.3f} {report['macro avg']['f1-score']:>10.3f}")
print(f"{'Weighted Avg':<30} {report['weighted avg']['precision']:>10.3f} "
      f"{report['weighted avg']['recall']:>10.3f} {report['weighted avg']['f1-score']:>10.3f}")

In [ ]:
# --- 17. Confusion Matrix (Top 20 most common classes) ---
# Show subset for readability (102 classes is too dense)
top_k = 20
class_counts = Counter(all_labels)
top_classes = [cls for cls, _ in class_counts.most_common(top_k)]
top_names = [FLOWER_NAMES[c] for c in top_classes]

# Filter to only top-K classes
mask = np.isin(all_labels, top_classes)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

# Remap to 0..top_k-1
label_map = {old: new for new, old in enumerate(top_classes)}
mapped_labels = np.array([label_map[l] for l in filtered_labels])
mapped_preds = np.array([label_map.get(p, -1) for p in filtered_preds])

# Only include valid mapped predictions
valid_mask = mapped_preds >= 0
cm = confusion_matrix(mapped_labels[valid_mask], mapped_preds[valid_mask], labels=range(top_k))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=top_names, yticklabels=top_names, ax=ax,
            linewidths=0.5, square=True)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Top {top_k} Classes)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 18. Per-Class Accuracy (Bottom & Top 10) ---
per_class_acc = {}
for cls_idx in range(NUM_CLASSES):
    mask = all_labels == cls_idx
    if mask.sum() > 0:
        per_class_acc[FLOWER_NAMES[cls_idx]] = (all_preds[mask] == cls_idx).mean()

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Bottom 10 (hardest classes)
ax = axes[0]
bottom_10 = sorted_acc[:10]
names = [x[0] for x in bottom_10]
accs = [x[1] * 100 for x in bottom_10]
bars = ax.barh(names, accs, color='salmon', edgecolor='darkred', alpha=0.8)
ax.set_xlabel('Accuracy (%)')
ax.set_title('❌ Bottom 10 — Hardest Classes', fontweight='bold')
ax.set_xlim(0, 100)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)

# Top 10 (easiest classes)
ax = axes[1]
top_10 = sorted_acc[-10:]
names = [x[0] for x in top_10]
accs = [x[1] * 100 for x in top_10]
bars = ax.barh(names, accs, color='lightgreen', edgecolor='darkgreen', alpha=0.8)
ax.set_xlabel('Accuracy (%)')
ax.set_title('✅ Top 10 — Easiest Classes', fontweight='bold')
ax.set_xlim(0, 105)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)

plt.suptitle('Per-Class Accuracy Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 19. Prediction Visualization ---
def show_predictions(model, dataset, class_names, device, n=16, ncols=4):
    """Display random predictions with confidence bars."""
    model.eval()
    indices = random.sample(range(len(dataset)), n)
    nrows = n // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))

    for i, ax in enumerate(axes.flat):
        image, label = dataset[indices[i]]
        img_display = denormalize(image).permute(1, 2, 0).numpy()

        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(device)
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)[0]
            pred_class = probs.argmax().item()
            confidence = probs[pred_class].item()

        # Top-3 predictions
        top3_probs, top3_idx = probs.topk(3)
        top3_text = "\n".join([f"  {class_names[idx]}: {p:.1%}"
                               for p, idx in zip(top3_probs.cpu(), top3_idx.cpu())])

        correct = pred_class == label
        color = 'green' if correct else 'red'
        symbol = '✅' if correct else '❌'

        ax.imshow(img_display)
        ax.set_title(f"{symbol} Pred: {class_names[pred_class]}\n"
                     f"True: {class_names[label]} ({confidence:.1%})",
                     fontsize=8, color=color, fontweight='bold')
        ax.axis('off')

    plt.suptitle('🔍 Random Test Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, FLOWER_NAMES, device, n=16)

## 🔬 Single Image Inference & MobileNetV3 Comparison

In [ ]:
# --- 20. Single Image Inference Function ---
def predict_flower(model, image_path_or_pil, class_names, device, transform=None, top_k=5):
    """
    Predict flower species from a single image.

    Args:
        model: Trained MobileNetV3 model
        image_path_or_pil: str path or PIL Image
        class_names: list of class names
        device: torch device
        transform: preprocessing transform
        top_k: number of top predictions to show

    Returns:
        dict with predictions and visualization
    """
    if transform is None:
        transform = val_transform

    # Load image
    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert('RGB')
    else:
        image = image_path_or_pil.convert('RGB')

    # Preprocess
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Inference
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]

    # Top-K predictions
    top_probs, top_indices = probs.topk(top_k)
    top_probs = top_probs.cpu().numpy()
    top_indices = top_indices.cpu().numpy()
    top_names = [class_names[i] for i in top_indices]

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                              gridspec_kw={'width_ratios': [1, 1.5]})

    # Original image
    axes[0].imshow(image)
    axes[0].set_title(f"🌸 Predicted: {top_names[0]} ({top_probs[0]:.1%})",
                      fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Probability bar chart
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(top_k)]
    bars = axes[1].barh(range(top_k-1, -1, -1), top_probs, color=colors, edgecolor='white')
    axes[1].set_yticks(range(top_k-1, -1, -1))
    axes[1].set_yticklabels(top_names, fontsize=10)
    axes[1].set_xlabel('Confidence')
    axes[1].set_title(f'Top-{top_k} Predictions', fontweight='bold')
    axes[1].set_xlim(0, 1.1)
    for bar, prob in zip(bars, top_probs):
        axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f'{prob:.2%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    return {
        'prediction': top_names[0],
        'confidence': float(top_probs[0]),
        'top_k': list(zip(top_names, top_probs.tolist())),
    }

# Demo: predict a random test image
sample_idx = random.randint(0, len(hf_dataset['test']) - 1)
sample_image = hf_dataset['test'][sample_idx]['image']
sample_label = hf_dataset['test'][sample_idx]['label']

print(f"Ground Truth: {FLOWER_NAMES[sample_label]}")
result = predict_flower(model, sample_image, FLOWER_NAMES, device)
print(f"\nPrediction:   {result['prediction']} ({result['confidence']:.2%})")

In [ ]:
# --- 21. MobileNetV3 Small vs Large Comparison ---
print("📊 MobileNetV3 Architecture Comparison")
print("=" * 75)

comparison_data = []
for variant, WeightsCls, ModelFn in [
    ('Small', models.MobileNet_V3_Small_Weights, models.mobilenet_v3_small),
    ('Large', models.MobileNet_V3_Large_Weights, models.mobilenet_v3_large),
]:
    m = ModelFn(weights=None)
    params = sum(p.numel() for p in m.parameters())
    size_mb = params * 4 / 1024**2

    # Measure inference speed (CPU)
    dummy = torch.randn(1, 3, 224, 224)
    m.eval()
    times = []
    with torch.no_grad():
        for _ in range(20):
            t0 = time.time()
            _ = m(dummy)
            times.append(time.time() - t0)
    avg_ms = np.mean(times[5:]) * 1000  # skip warmup

    comparison_data.append({
        'Variant': f'MobileNetV3-{variant}',
        'Parameters': f'{params/1e6:.2f}M',
        'Size (FP32)': f'{size_mb:.1f} MB',
        'CPU Latency': f'{avg_ms:.1f} ms',
        'Last Channel': 576 if variant == 'Small' else 960,
    })
    del m

# Display comparison
print(f"{'Metric':<20} {'MobileNetV3-Small':>20} {'MobileNetV3-Large':>20}")
print("-" * 62)
for key in ['Parameters', 'Size (FP32)', 'CPU Latency', 'Last Channel']:
    print(f"{key:<20} {comparison_data[0][key]:>20} {comparison_data[1][key]:>20}")

print("\n" + "-" * 62)
print("📌 Key Differences:")
print("   • Small: Optimized for latency — fewer inverted residuals, reduced channels")
print("   • Large: Higher accuracy — more layers, wider channels, same SE + h-swish")
print("   • Both use NAS-searched architectures with NetAdapt refinement")
print("   • Both support depthwise separable convolutions + squeeze-excitation")

In [ ]:
# --- 22. Export to ONNX (Mobile Deployment Ready) ---
print("📱 Exporting to ONNX for Mobile/Edge Deployment...")

model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
onnx_path = f"mobilenetv3_{VARIANT}_flowers102.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size'},
        'output': {0: 'batch_size'},
    },
)

onnx_size = os.path.getsize(onnx_path) / 1024**2
pytorch_size = os.path.getsize(SAVE_PATH) / 1024**2

print(f"✅ ONNX model exported: {onnx_path}")
print(f"   ONNX size:    {onnx_size:.1f} MB")
print(f"   PyTorch size: {pytorch_size:.1f} MB")
print(f"\n💡 Deployment Options:")
print(f"   • Android/iOS: ONNX Runtime Mobile or TFLite (via onnx-tf)")
print(f"   • Edge/IoT:    ONNX Runtime + quantization (INT8)")
print(f"   • Web:         ONNX.js or ONNX Runtime Web (WebAssembly)")
print(f"   • Server:      TorchServe or Triton Inference Server")

## 📝 Summary

### Results
| Metric | Value |
|--------|-------|
| **Model** | MobileNetV3-Large (Transfer Learning) |
| **Dataset** | Oxford Flowers 102 (102 classes) |
| **Training** | 2-Phase (Frozen → Fine-tune) |
| **Best Val Accuracy** | Check training logs above |
| **ONNX Export** | ✅ Mobile deployment ready |

### MobileNetV3 Key Takeaways
1. **Efficient Architecture** — 5.4M params (Large) vs 2.5M (Small), designed for mobile
2. **h-swish Activation** — Hardware-efficient alternative to Swish
3. **SE Blocks** — Channel attention with minimal compute overhead
4. **NAS + NetAdapt** — Architecture search + per-layer optimization
5. **Last Stage Redesign** — Expensive operations moved after global pooling
6. **Transfer Learning** — 2-phase strategy prevents catastrophic forgetting
7. **ONNX Export** — Enables cross-platform mobile/edge deployment

### References
- [Searching for MobileNetV3 (Howard et al., 2019)](https://arxiv.org/abs/1905.02244)
- [MobileNetV2: Inverted Residuals (Sandler et al., 2018)](https://arxiv.org/abs/1801.04381)
- [PyTorch MobileNetV3 Docs](https://pytorch.org/vision/stable/models/mobilenetv3.html)
- [Oxford Flowers 102 Dataset](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/)